# 消费信贷申请欺诈风险特征分析与风险分层｜Python / Pandas

## 项目背景
本模块基于模拟消费信贷申请数据，分析申请阶段可能与欺诈风险相关的行为和身份特征，并将多个风险信号组合为可解释的规则命中数和风险等级。

## 分析目标
1. 检查申请数据质量和整体欺诈率；
2. 分析高频申请、身份信息一致性、申请地区、新设备和信用历史等风险特征；
3. 将单一风险特征统一转换为0/1风险信号；
4. 统计每笔申请命中的风险规则数量；
5. 构建低、中、高欺诈风险分层，并输出最终风险结果表。

## 分析框架
`数据检查 → 单特征欺诈分析 → 风险特征工程 → 多规则组合 → 风险分层 → 结果输出`

> 本模块使用模拟申请欺诈数据，用于展示反欺诈特征分析和规则策略构建流程，不代表真实金融机构生产环境中的欺诈率或正式拦截规则。

## 1. 数据读取与基础检查

一行数据代表一笔消费信贷申请，目标变量为：

- `fraud_flag = 0`：正常申请
- `fraud_flag = 1`：欺诈申请

身份证号和手机号属于标识符字段，不作为连续数值变量参与本项目分析，因此统一转为字符串类型。

In [ ]:
import pandas as pd

fraud_df = pd.read_csv("data/fraud_df.csv")

fraud_df["id_number"] = fraud_df["id_number"].astype(str)
fraud_df["phone_number"] = fraud_df["phone_number"].astype(str)

In [ ]:
print("数据规模：", fraud_df.shape)
print("字段数量：", len(fraud_df.columns))
print("字段名称：", fraud_df.columns.tolist())

fraud_df.head()

数据规模： (2000, 14)
字段数量： 14
字段名称： ['application_id', 'apply_time', 'id_number', 'phone_number', 'device_id', 'ip_address', 'apply_region', 'apply_channel', 'loan_amount', 'apply_freq_7d', 'id_consistency', 'is_new_device', 'credit_history', 'fraud_flag']


,application_id,apply_time,id_number,phone_number,device_id,ip_address,apply_region,apply_channel,loan_amount,apply_freq_7d,id_consistency,is_new_device,credit_history,fraud_flag
0,APP000001,2026-07-21 03:01:00,513813795610077732,13955030011,DEV_147325,223.104.90.26,湖南,APP,13561.50,1,1,0,有记录,0
1,APP000002,2026-07-24 08:15:00,513043884302178167,13205806587,DEV_520535,103.17.47.31,浙江,H5,43978.77,10,1,1,无记录,1
2,APP000003,2026-07-08 04:47:00,511937455611404288,15662990246,DEV_344847,192.168.224.100,广东,线下,17834.16,2,1,1,有记录,0
3,APP000004,2026-07-04 21:47:00,516596789760987312,17673927597,DEV_313716,10.0.120.143,四川,APP,10119.73,2,1,0,有记录,0
4,APP000005,2026-07-29 17:05:00,519893201265789826,19743012179,DEV_129440,192.168.122.194,北京,APP,13474.30,2,1,0,无记录,0


### 1.1 数据质量检查

检查缺失值和完全重复记录，确认数据是否可以直接进入分析阶段。

In [ ]:
print("缺失值总数：", fraud_df.isna().sum().sum())
print("完全重复行数量：", fraud_df.duplicated().sum())

缺失值总数： 0
完全重复行数量： 0


### 1.2 整体欺诈情况

由于 `fraud_flag` 是0/1变量，因此：

`fraud_df["fraud_flag"].mean()`

可以直接表示整体欺诈率。

In [ ]:
fraud_count = fraud_df["fraud_flag"].value_counts().sort_index()
fraud_rate = fraud_df["fraud_flag"].mean()

print("正常申请：", fraud_count[0])
print("欺诈申请：", fraud_count[1])
print("整体欺诈率：", fraud_rate)

正常申请： 1803
欺诈申请： 197
整体欺诈率： 0.0985


**数据概览：**
- 总申请数：2,000
- 正常申请：1,803
- 欺诈申请：197
- 整体欺诈率：9.85%
- 数据无缺失值、无完全重复行

后续分析重点关注不同申请特征下的欺诈率差异。

## 2. 近7天申请频率与欺诈风险

**分析问题：**
短期内频繁提交申请的客户，欺诈率是否更高？

先观察 `apply_freq_7d` 的分布，再计算不同申请次数对应的欺诈率。

In [ ]:
fraud_df["apply_freq_7d"].describe()

count    2000.000000
mean        2.708000
std         2.489143
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        15.000000
Name: apply_freq_7d, dtype: float64

In [ ]:
application_freq_count = fraud_df["apply_freq_7d"].value_counts().sort_index()
application_freq_fraud_rate = fraud_df.groupby("apply_freq_7d")["fraud_flag"].mean().sort_index()

application_freq_summary = pd.DataFrame({
    "申请数量": application_freq_count,
    "欺诈率": application_freq_fraud_rate
})

application_freq_summary

,申请数量,欺诈率
apply_freq_7d,,
1,558,0.000000
2,632,0.000000
3,631,0.028526
4,11,1.000000
5,21,1.000000
6,16,1.000000
7,12,1.000000
8,14,1.000000
9,11,1.000000


**分析结果：**
- 近7天申请1次和2次的申请欺诈率为0；
- 申请3次的欺诈率约为2.85%；
- 申请4次及以上的样本在当前模拟数据中欺诈率为100%。

在该数据中，3次到4次之间出现非常明显的风险断点，因此将：

`apply_freq_7d >= 4`

定义为“高频申请”风险信号。

> 4次及以上欺诈率达到100%是该模拟数据的特征，不能直接解释为真实业务中的通用阈值。

In [ ]:
fraud_df["high_freq_flag"] = (fraud_df["apply_freq_7d"] >= 4).astype(int)

high_freq_count = fraud_df["high_freq_flag"].value_counts().sort_index()
high_freq_fraud_rate = fraud_df.groupby("high_freq_flag")["fraud_flag"].mean()

high_freq_summary = pd.DataFrame({
    "申请数量": high_freq_count,
    "欺诈率": high_freq_fraud_rate
})

high_freq_summary

,申请数量,欺诈率
high_freq_flag,,
0,1821,0.009885
1,179,1.000000


**风险信号定义：**
- `high_freq_flag = 1`：近7天申请次数 ≥ 4
- `high_freq_flag = 0`：近7天申请次数 < 4

高频申请组共179条申请，在当前模拟数据中的欺诈率为100%；非高频申请组欺诈率约为0.99%。

## 3. 身份信息一致性与欺诈风险

**分析问题：**
身份信息不一致的申请是否具有更高的欺诈风险？

原字段含义：
- `id_consistency = 1`：身份信息一致
- `id_consistency = 0`：身份信息不一致

In [ ]:
id_consistency_count = fraud_df["id_consistency"].value_counts().sort_index()
id_consistency_fraud_rate = fraud_df.groupby("id_consistency")["fraud_flag"].mean()

id_consistency_summary = pd.DataFrame({
    "申请数量": id_consistency_count,
    "欺诈率": id_consistency_fraud_rate
})

id_consistency_summary

,申请数量,欺诈率
id_consistency,,
0,66,1.000000
1,1934,0.067735


**分析结果：**
- 身份信息一致：1,934条，欺诈率约6.77%
- 身份信息不一致：66条，欺诈率100%

为了使所有风险变量方向统一为“1 = 风险存在”，进一步构造 `id_inconsistent_flag`。

In [ ]:
fraud_df["id_inconsistent_flag"] = (fraud_df["id_consistency"] == 0).astype(int)

fraud_df["id_inconsistent_flag"].value_counts().sort_index()

id_inconsistent_flag
0    1934
1      66
Name: count, dtype: int64

## 4. 申请地区与欺诈风险

**分析问题：**
不同申请地区的欺诈率是否存在明显差异？

先统计各地区申请数量和欺诈率，并按欺诈率从高到低观察。

In [ ]:
region_count = fraud_df["apply_region"].value_counts()
region_fraud_rate = fraud_df.groupby("apply_region")["fraud_flag"].mean().sort_values(ascending=False)

region_summary = pd.DataFrame({
    "申请数量": region_count,
    "欺诈率": region_fraud_rate
})

region_summary.sort_values("欺诈率", ascending=False)

,申请数量,欺诈率
apply_region,,
未知地区,20,1.000000
境外,16,1.000000
江苏,256,0.109375
浙江,238,0.100840
广东,246,0.097561
北京,229,0.082969
四川,252,0.079365
上海,229,0.069869
湖南,265,0.060377


**分析结果：**
`未知地区`和`境外`申请在当前模拟数据中的欺诈率均为100%，明显高于其他地区。

因此构造异常地区风险信号：
- `apply_region` 为“境外”或“未知地区” → 1
- 其他地区 → 0

In [ ]:
fraud_df["abnormal_region_flag"] = fraud_df["apply_region"].isin(
    ["境外", "未知地区"]
).astype(int)

fraud_df["abnormal_region_flag"].value_counts().sort_index()

abnormal_region_flag
0    1964
1      36
Name: count, dtype: int64

## 5. 新设备申请与欺诈风险

**分析问题：**
使用新设备提交申请，是否与更高的欺诈风险相关？

`is_new_device` 本身已经是0/1变量：
- `1`：新设备
- `0`：非新设备

因此不需要额外重新编码。

In [ ]:
new_device_count = fraud_df["is_new_device"].value_counts().sort_index()
new_device_fraud_rate = fraud_df.groupby("is_new_device")["fraud_flag"].mean()

new_device_summary = pd.DataFrame({
    "申请数量": new_device_count,
    "欺诈率": new_device_fraud_rate
})

new_device_summary

,申请数量,欺诈率
is_new_device,,
0,1140,0.041228
1,860,0.174419


**分析结果：**
- 非新设备申请：1,140条，欺诈率约4.12%
- 新设备申请：860条，欺诈率约17.44%

新设备申请的欺诈率约为非新设备的4.23倍，说明新设备是一个具有明显区分度的风险特征。

## 6. 信用历史与欺诈风险

**分析问题：**
缺少信用历史的申请是否表现出更高的欺诈风险？

原字段为文本类别：
- `有记录`
- `无记录`

In [ ]:
credit_history_count = fraud_df["credit_history"].value_counts()
credit_history_fraud_rate = fraud_df.groupby("credit_history")["fraud_flag"].mean()

credit_history_summary = pd.DataFrame({
    "申请数量": credit_history_count,
    "欺诈率": credit_history_fraud_rate
})

credit_history_summary

,申请数量,欺诈率
credit_history,,
无记录,689,0.171263
有记录,1311,0.060259


**分析结果：**
- 有信用记录：1,311条，欺诈率约6.03%
- 无信用记录：689条，欺诈率约17.13%

为了统一风险方向，将“无信用记录”映射为1，“有信用记录”映射为0。

In [ ]:
fraud_df["no_credit_history_flag"] = fraud_df["credit_history"].map({
    "无记录": 1,
    "有记录": 0
})

fraud_df["no_credit_history_flag"].value_counts().sort_index()

no_credit_history_flag
0    1311
1     689
Name: count, dtype: int64

## 7. 欺诈风险特征工程

经过单特征分析，形成5个方向统一的风险信号：

| 风险特征 | 风险条件 | 取值方向 |
|---|---|---|
| `high_freq_flag` | 近7天申请次数 ≥ 4 | 1 = 风险 |
| `id_inconsistent_flag` | 身份信息不一致 | 1 = 风险 |
| `abnormal_region_flag` | 境外 / 未知地区 | 1 = 风险 |
| `is_new_device` | 使用新设备申请 | 1 = 风险 |
| `no_credit_history_flag` | 无信用记录 | 1 = 风险 |

统一风险方向后，可以直接统计每笔申请同时命中了多少项风险规则。

In [ ]:
risk_rule_columns = [
    "high_freq_flag",
    "id_inconsistent_flag",
    "abnormal_region_flag",
    "is_new_device",
    "no_credit_history_flag"
]

fraud_df["fraud_rule_count"] = fraud_df[risk_rule_columns].sum(axis=1)

fraud_df["fraud_rule_count"].value_counts().sort_index()

fraud_rule_count
0    736
1    871
2    261
3     93
4     37
5      2
Name: count, dtype: int64

## 8. 多规则命中与欺诈风险

**分析问题：**
一笔申请同时命中的风险规则越多，欺诈率是否越高？

按照 `fraud_rule_count` 分组，比较不同规则命中数量下的申请数量和欺诈率。

In [ ]:
fraud_rule_count = fraud_df["fraud_rule_count"].value_counts().sort_index()
fraud_rule_fraud_rate = fraud_df.groupby("fraud_rule_count")["fraud_flag"].mean().sort_index()

fraud_rule_summary = pd.DataFrame({
    "申请数量": fraud_rule_count,
    "欺诈率": fraud_rule_fraud_rate
})

fraud_rule_summary

,申请数量,欺诈率
fraud_rule_count,,
0,736,0.000000
1,871,0.020666
2,261,0.180077
3,93,1.000000
4,37,1.000000
5,2,1.000000


**分析结果：**
- 命中0条规则：欺诈率0%
- 命中1条规则：欺诈率约2.07%
- 命中2条规则：欺诈率约18.01%
- 命中3条及以上：在当前模拟数据中欺诈率为100%

整体上，随着风险规则命中数量增加，欺诈率明显上升，说明多风险信号叠加具有较强的风险区分能力。

需要注意的是，命中5条规则的样本仅有2条；同时3条以上达到100%的结果具有明显的模拟数据特征，因此不能将该结果直接解释为真实生产环境中的确定性欺诈规则。

## 9. 欺诈风险分层

根据规则命中数量进行探索性风险分层：

- **低风险：** 命中0-1条规则
- **中风险：** 命中2条规则
- **高风险：** 命中3条及以上规则

该分层根据当前模拟数据中的欺诈率变化设置，用于展示规则策略思路，不代表真实业务中的正式拦截阈值。

In [ ]:
fraud_df["fraud_risk_level"] = pd.cut(
    fraud_df["fraud_rule_count"],
    bins=[-1, 1, 2, float("inf")],
    labels=["低风险", "中风险", "高风险"],
    right=True
)

risk_level_count = fraud_df["fraud_risk_level"].value_counts().sort_index()
risk_level_fraud_rate = fraud_df.groupby(
    "fraud_risk_level",
    observed=False
)["fraud_flag"].mean()

risk_level_summary = pd.DataFrame({
    "申请数量": risk_level_count,
    "欺诈率": risk_level_fraud_rate
})

risk_level_summary

,申请数量,欺诈率
fraud_risk_level,,
低风险,1607,0.011201
中风险,261,0.180077
高风险,132,1.000000


**风险分层结果：**
- 低风险：1,607条，欺诈率约1.12%
- 中风险：261条，欺诈率约18.01%
- 高风险：132条，欺诈率100%

三个风险层级呈现出明显的欺诈率梯度，说明组合风险规则能够有效区分不同风险客群。

> 高风险组100%的结果来自当前模拟数据，需要通过真实业务样本、样本外数据和误伤率等指标进一步验证。

## 10. 最终风险结果表

保留申请编号、5项风险信号、规则命中数、风险等级和真实欺诈标签，形成最终风险结果表。

该表可以用于后续策略复盘、风险客户筛选和规则效果验证。

In [ ]:
fraud_result = fraud_df[[
    "application_id",
    "high_freq_flag",
    "id_inconsistent_flag",
    "abnormal_region_flag",
    "is_new_device",
    "no_credit_history_flag",
    "fraud_rule_count",
    "fraud_risk_level",
    "fraud_flag"
]].copy()

fraud_result.head()

,application_id,high_freq_flag,id_inconsistent_flag,abnormal_region_flag,is_new_device,no_credit_history_flag,fraud_rule_count,fraud_risk_level,fraud_flag
0,APP000001,0,0,0,0,0,0,低风险,0
1,APP000002,1,0,0,1,1,3,高风险,1
2,APP000003,0,0,0,1,0,1,低风险,0
3,APP000004,0,0,0,0,0,0,低风险,0
4,APP000005,0,0,0,0,1,1,低风险,0


In [ ]:
fraud_result.to_csv(
    "outputs/fraud_result.csv",
    index=False,
    encoding="utf-8-sig"
)

## 11. 核心发现与风险启示

1. **短期高频申请是当前数据中最强的欺诈风险信号之一。** 近7天申请4次及以上的样本欺诈率显著高于非高频申请。
2. **身份信息异常和异常地区具有很强的风险区分能力。** 身份信息不一致、境外或未知地区申请在当前模拟数据中均表现出极高欺诈率。
3. **新设备和缺少信用历史属于概率型风险信号。** 两类特征并不意味着申请一定欺诈，但对应客群欺诈率明显更高，更适合作为组合规则的一部分。
4. **多风险信号叠加后区分度进一步增强。** 规则命中数量从0增加到2时，欺诈率从0%上升到约18.01%；3条及以上在模拟数据中达到100%。
5. **风险规则需要组合判断。** 单一特征适合用于风险提示，多特征组合更适合形成可解释的风险分层和人工审核优先级。

## 12. 项目局限性

- 本模块使用模拟申请欺诈数据，数据规律比真实业务环境更加清晰；
- 高频申请、身份异常和异常地区在部分组别出现100%欺诈率，不能据此认定真实业务中对应条件必然代表欺诈；
- 当前风险规则阈值主要基于本数据集的探索结果设置，没有进行训练集/测试集划分和样本外验证；
- 本项目未纳入设备共享网络、IP关联、手机号复用、团伙关系图谱等更复杂的反欺诈特征；
- 当前分析重点是可解释的规则策略，没有构建机器学习欺诈识别模型；
- 如果用于真实业务，还需要进一步评估召回率、误伤率、通过率、人工审核成本和欺诈损失等业务指标。

**项目定位：** 消费信贷申请欺诈风险特征分析与可解释规则分层项目，重点展示 Pandas 数据分析、欺诈特征工程、规则组合和风险策略思维。